# Albanian NER: fine-tuned encoder baselines

This notebook fine-tunes **XLM-R**, **mDeBERTa-v3** and **Glot500** and scores them on the gold test set
(693 sentences from parliamentary transcripts, laws and government reports; 691 after SKIP).

| Training data | Types | What it shows |
|---|---|---|
| **WikiANN-sq** (5,000 Wikipedia sentences, downloaded automatically) | PER, ORG, LOC | The standard baseline: how well models trained on Wikipedia transfer to administrative text |
| **AlbNER** (500 train sentences, optional upload) | PER, ORG, LOC, MISC | The only other gold Albanian corpus with MISC |

The gold test set is **never** used for training or for choosing the best epoch: each model picks its best epoch on
the validation split of its own training data. Every configuration runs with several seeds; report mean ± standard deviation.

**Before you start:** *Runtime -> Change runtime type -> T4 GPU*, then *Runtime -> Run all*.

In [ ]:
!pip -q install seqeval sentencepiece protobuf
import transformers, datasets, torch
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 1. Upload the data

Upload **`test.jsonl`** (from `CORPUS/bio/`, made by `export_to_bio.py`).

Optional, for the AlbNER runs: download AlbNER from [CLARIN](http://hdl.handle.net/11234/1-5214), unzip it on your
computer, and upload its **train**, **dev** and **test** files in the same dialog. Any file whose name contains
"train", "dev"/"valid" or "test" (and is not `test.jsonl`) is treated as AlbNER.

In [ ]:
import os, shutil
os.makedirs("/content/data", exist_ok=True)
try:
    from google.colab import files
    for name, content in files.upload().items():
        open(os.path.join("/content/data", name), "wb").write(content)
except ImportError:
    pass
print(sorted(os.listdir("/content/data")))

## 2. Settings

The defaults take about **1.5-2 hours** on a T4 for everything (3 models x 3 seeds x WikiANN, plus AlbNER if
uploaded). For a quick first test, set `SEEDS = [1]`.

In [ ]:
MODELS = {
    "XLM-R base":       "FacebookAI/xlm-roberta-base",
    "mDeBERTa-v3 base": "microsoft/mdeberta-v3-base",
    "Glot500 base":     "cis-lmu/glot500-base",
}
SEEDS = [1, 2, 3]
TRAIN_SETS = ["wikiann", "albner"]
EPOCHS = {"wikiann": 5, "albner": 20}
LR, BATCH, MAX_LEN = 3e-5, 16, 256
OUT = "/content/results"
os.makedirs(f"{OUT}/predictions", exist_ok=True)

## 3. Data loading

In [ ]:
import json, glob, re
from collections import Counter
from datasets import Dataset, load_dataset

TYPES4 = ["PER", "ORG", "LOC", "MISC"]
ALIASES = {"PERSON": "PER", "PERS": "PER", "ORGANIZATION": "ORG", "ORGANISATION": "ORG",
           "LOCATION": "LOC", "GPE": "LOC", "MISCELLANEOUS": "MISC", "OTH": "MISC", "OTHER": "MISC"}

def clean_tags(tags, keep=TYPES4):
    out, prev = [], "O"
    for t in tags:
        t = str(t).strip().upper().replace("_", "-")
        if t in ("O", "", "0"):
            out.append("O"); prev = "O"; continue
        if "-" not in t:
            t = "I-" + t
        p, typ = t.split("-", 1)
        typ = ALIASES.get(typ, typ)
        if typ not in keep:
            out.append("O"); prev = "O"; continue
        if p == "I" and prev not in ("B-" + typ, "I-" + typ):
            p = "B"
        if p not in ("B", "I"):
            p = "B"
        out.append(f"{p}-{typ}"); prev = out[-1]
    return out

def load_gold(path="/content/data/test.jsonl"):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

def load_wikiann():
    try:
        ds = load_dataset("unimelb-nlp/wikiann", "sq")
    except Exception as e:
        print("Standard load failed (", type(e).__name__, "), trying the parquet branch")
        base = "hf://datasets/unimelb-nlp/wikiann@refs%2Fconvert%2Fparquet/sq"
        ds = load_dataset("parquet", data_files={s: f"{base}/{s}/*.parquet" for s in ("train", "validation", "test")})
    names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
    f = ds["train"].features["ner_tags"]
    if hasattr(f, "feature") and hasattr(f.feature, "names"):
        names = f.feature.names
    conv = lambda split: [{"tokens": r["tokens"], "ner_tags": clean_tags([names[i] for i in r["ner_tags"]])}
                          for r in ds[split]]
    return {"train": conv("train"), "validation": conv("validation"), "test": conv("test")}

def read_conll(path):
    sents, toks, tags = [], [], []
    for line in open(path, encoding="utf-8", errors="replace"):
        line = line.strip()
        if not line or line.startswith("-DOCSTART-") or line.startswith("#"):
            if toks:
                sents.append({"tokens": toks, "ner_tags": clean_tags(tags)}); toks, tags = [], []
            continue
        parts = line.split("\t") if "\t" in line else line.split()
        if len(parts) < 2:
            continue
        toks.append(parts[0]); tags.append(parts[-1])
    if toks:
        sents.append({"tokens": toks, "ner_tags": clean_tags(tags)})
    return sents

def load_albner(folder="/content/data"):
    files = [f for f in glob.glob(f"{folder}/*") if os.path.basename(f) != "test.jsonl"
             and os.path.isfile(f) and not f.endswith((".zip", ".pdf", ".md"))]
    pick = lambda *keys: next((f for f in files if any(k in os.path.basename(f).lower() for k in keys)), None)
    tr, dv, te = pick("train"), pick("dev", "valid"), pick("test")
    if not (tr and dv):
        return None
    return {"train": read_conll(tr), "validation": read_conll(dv), "test": read_conll(te) if te else []}

def describe(name, d):
    for split, rows in d.items():
        c = Counter(t[2:] for r in rows for t in r["ner_tags"] if t.startswith("B-"))
        print(f"  {name:8} {split:10} {len(rows):5} sentences  " + "  ".join(f"{k}={c[k]}" for k in TYPES4))

In [ ]:
gold = load_gold()
data = {"wikiann": load_wikiann()}
alb = load_albner()
if alb:
    data["albner"] = alb
else:
    print("AlbNER files not found: AlbNER runs are skipped")
print(f"gold test: {len(gold)} sentences")
for k, v in data.items():
    describe(k, v)

## 4. Training and evaluation

In [ ]:
import gc, inspect, numpy as np, pandas as pd, torch
from seqeval.metrics import classification_report, f1_score
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments, set_seed)

def label_list(train_rows):
    types = sorted({t[2:] for r in train_rows for t in r["ner_tags"] if t != "O"}, key=TYPES4.index)
    return ["O"] + [f"{p}-{t}" for t in types for p in ("B", "I")], types

def encode(rows, tok, l2i, max_len):
    enc = tok([r["tokens"] for r in rows], is_split_into_words=True, truncation=True, max_length=max_len)
    labels = []
    for i, r in enumerate(rows):
        prev, lab = None, []
        for w in enc.word_ids(i):
            lab.append(-100 if w is None or w == prev else l2i.get(r["ner_tags"][w], 0))
            prev = w
        labels.append(lab)
    enc["labels"] = labels
    return Dataset.from_dict(dict(enc))

def decode(pred_logits, rows, enc_ds, tok, i2l, max_len):
    enc = tok([r["tokens"] for r in rows], is_split_into_words=True, truncation=True, max_length=max_len)
    ids = pred_logits.argmax(-1)
    out = []
    for i, r in enumerate(rows):
        tags, prev = ["O"] * len(r["tokens"]), None
        for j, w in enumerate(enc.word_ids(i)):
            if w is not None and w != prev:
                tags[w] = i2l[int(ids[i][j])]
            prev = w
        out.append(clean_tags(tags))
    return out

def scores(gold_tags, pred_tags, types):
    rep = classification_report(gold_tags, pred_tags, output_dict=True, zero_division=0)
    s = {"micro_F1": rep["micro avg"]["f1-score"], "macro_F1": rep["macro avg"]["f1-score"],
         "P": rep["micro avg"]["precision"], "R": rep["micro avg"]["recall"]}
    for t in types:
        s[f"F1_{t}"] = rep.get(t, {}).get("f1-score", 0.0)
    return s

def run(model_key, train_key, seed):
    model_id, d = MODELS[model_key], data[train_key]
    labels, types = label_list(d["train"])
    l2i, i2l = {l: i for i, l in enumerate(labels)}, dict(enumerate(labels))
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id, num_labels=len(labels),
                                                            id2label=i2l, label2id=l2i, ignore_mismatched_sizes=True)
    model = model.float()
    tr, dv = encode(d["train"], tok, l2i, MAX_LEN), encode(d["validation"], tok, l2i, MAX_LEN)

    def dev_f1(p):
        logits, lab = p
        ids = logits.argmax(-1)
        g = [[i2l[l] for l in row if l != -100] for row in lab]
        pr = [[i2l[int(x)] for x, l in zip(prow, row) if l != -100] for prow, row in zip(ids, lab)]
        return {"f1": f1_score(g, pr, zero_division=0)}

    out_dir = f"/content/ckpt/{train_key}_{model_key.replace(' ', '_')}_{seed}"
    warmup = int(0.1 * EPOCHS[train_key] * -(-len(tr) // BATCH))
    kw = dict(
        output_dir=out_dir, learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=32,
        num_train_epochs=EPOCHS[train_key], weight_decay=0.01, warmup_steps=warmup,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        fp16=torch.cuda.is_available() and "deberta" not in model_id.lower(),
        logging_steps=50, report_to="none", seed=seed, dataloader_num_workers=2)
    accepted = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in accepted:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    args = TrainingArguments(**{k: v for k, v in kw.items() if k in accepted})
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=dv,
                      data_collator=DataCollatorForTokenClassification(tok), compute_metrics=dev_f1)
    trainer.train()
    os.makedirs(f"{OUT}/logs", exist_ok=True)
    log = trainer.state.log_history
    json.dump(log, open(f"{OUT}/logs/{train_key}__{model_key.replace(' ', '_')}__seed{seed}.json", "w"), indent=1)
    losses = [x["loss"] for x in log if "loss" in x]
    devf1 = [round(x["eval_f1"], 3) for x in log if "eval_f1" in x]
    print("train loss:", [round(l, 3) for l in losses[:3]], "...", [round(l, 3) for l in losses[-2:]], "| dev F1 per epoch:", devf1)
    if not devf1 or max(devf1) == 0 or any(l != l for l in losses):
        print("WARNING: this run did not learn (dev F1 = 0 or NaN loss). Its scores are not valid.")

    results = []
    targets = [("gold", gold)] + ([(f"{train_key}_test", d["test"])] if d.get("test") else [])
    for tname, rows in targets:
        rows_eval = [dict(r, ner_tags=clean_tags(r["ner_tags"], keep=types)) for r in rows]
        enc = encode(rows_eval, tok, l2i, 512)
        logits = trainer.predict(enc).predictions
        pred = decode(logits, rows_eval, enc, tok, i2l, 512)
        g = [r["ner_tags"] for r in rows_eval]
        base = {"model": model_key, "train": train_key, "seed": seed, "test": tname, "types": "+".join(types)}
        results.append({**base, "subset": "all", "n": len(rows_eval), **scores(g, pred, types)})
        if tname == "gold":
            for key in ("country", "doc_type"):
                for val in sorted({r[key] for r in rows_eval}):
                    idx = [i for i, r in enumerate(rows_eval) if r[key] == val]
                    results.append({**base, "subset": f"{key}={val}", "n": len(idx),
                                    **scores([g[i] for i in idx], [pred[i] for i in idx], types)})
            if "MISC" not in types:
                full = [clean_tags(r["ner_tags"]) for r in rows]
                pn = []
                for ft, p in zip(full, pred):
                    p = list(p)
                    for i, t in enumerate(ft):
                        if t.endswith("-MISC"):
                            p[i] = "O"
                    pn.append(clean_tags(p, keep=types))
                results.append({**base, "subset": "all (MISC spans ignored)", "n": len(pn), **scores(g, pn, types)})
            if len(types) == 4:
                g3 = [clean_tags(x, keep=types[:3]) for x in g]; p3 = [clean_tags(x, keep=types[:3]) for x in pred]
                results.append({**base, "types": "PER+ORG+LOC", "subset": "all (3 types)", "n": len(g3),
                                **scores(g3, p3, types[:3])})
            with open(f"{OUT}/predictions/{train_key}__{model_key.replace(' ', '_')}__seed{seed}.jsonl", "w") as fh:
                for r, p in zip(rows_eval, pred):
                    fh.write(json.dumps({"id": r["id"], "country": r["country"], "doc_type": r["doc_type"],
                                         "tokens": r["tokens"], "gold": r["ner_tags"], "pred": p},
                                        ensure_ascii=False) + "\n")
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return results

In [ ]:
import time
all_rows, csv_path = [], f"{OUT}/results_all_runs.csv"
for train_key in [t for t in TRAIN_SETS if t in data]:
    for model_key in MODELS:
        for seed in SEEDS:
            t0 = time.time()
            print(f"\n=== {train_key} | {model_key} | seed {seed} ===")
            rows = run(model_key, train_key, seed)
            all_rows += rows
            pd.DataFrame(all_rows).to_csv(csv_path, index=False)
            r = rows[0]
            print(f"gold micro-F1 = {r['micro_F1']:.3f}  ({(time.time() - t0) / 60:.1f} min)")

## 5. Results (mean ± standard deviation over seeds)

In [ ]:
df = pd.read_csv(csv_path)
metric_cols = [c for c in df.columns if c in ("micro_F1", "macro_F1", "P", "R") or c.startswith("F1_")]

def summary(frame):
    g = frame.groupby(["train", "model", "test", "types", "subset"])[metric_cols]
    m, s = g.mean(), g.std().fillna(0)
    return (m.round(3).astype(str) + " ± " + s.round(3).astype(str)).reset_index()

MAIN = ["all", "all (3 types)", "all (MISC spans ignored)"]
main = summary(df[df.subset.isin(MAIN)])
by_sub = summary(df[df.test.eq("gold") & ~df.subset.isin(MAIN)])
main.to_csv(f"{OUT}/results_summary.csv", index=False)
by_sub.to_csv(f"{OUT}/results_by_country_doctype.csv", index=False)
pd.set_option("display.max_columns", None, "display.width", 250)
print("Main table: 'gold' = your test set; 'wikiann_test' / 'albner_test' = the training corpus's own test set")
display(main)
print("\nGold test set by country and document type")
display(by_sub[["train", "model", "subset", "micro_F1"]])

In [ ]:
shutil.make_archive("/content/ner_results", "zip", OUT)
try:
    from google.colab import files
    files.download("/content/ner_results.zip")
except ImportError:
    print("Results in", OUT)

**What to send back:** `ner_results.zip` contains `results_summary.csv`, `results_by_country_doctype.csv`,
`results_all_runs.csv` and `predictions/` (every gold sentence with gold and predicted tags, for the error analysis).